# Entrega 2 - Modelos de Linguagem e Busca Semântica

**Curso:** Processamento de Linguagem Natural  
**Aluno:** [Seu Nome Aqui]  
**Data:** [Data Atual]

---

## Contexto e objetivo

Você irá implementar componentes centrais de um **chatbot de e-commerce**:

1. **Modelo de linguagem por bigramas** para estimar probabilidade de frases e gerar sequências simples de texto.
2. **Busca em FAQ por similaridade** (TF-IDF + similaridade cosseno) para recuperar respostas a perguntas de usuários.

O que você implementar aqui **será reutilizado cumulativamente** nas próximas entregas e no Projeto Final. Portanto, trate cada função como parte de uma API estável: assinaturas, nomes e tipos esperados devem ser preservados.

---

## Instruções

1. Complete todos os exercícios marcados com `# === SEU CÓDIGO AQUI ===`
2. **Não modifique** os nomes das funções - eles serão usados no Projeto Final
3. Execute todas as células em ordem antes de enviar
4. **Testes ocultos** serão executados na correção automática

---

## Critérios de Avaliação

| Questão | Pontos | Descrição |
|---------|--------|------------|
| Q1 | 10 | Construir vocabulário |
| Q2 | 10 | Modelo bigrama |
| Q3 | 10 | Probabilidade bigrama |
| Q4 | 10 | Geração de texto |
| Q5 | 10 | Vetorizador TF-IDF |
| Q6 | 10 | Vetorizar texto |
| Q7 | 10 | Similaridade cosseno |
| Q8 | 15 | Busca FAQ (**CRÍTICO**) |
| Q9 | 15 | Sistema FAQ completo |
| **Total** | **100** | |

---

## Conteúdo Avaliado

- **Aula 3:** Modelos de linguagem N-grama
- **Aula 4:** Representação de documentos (Bag of Words, TF-IDF)

---

## Funções a Implementar

Implemente as funções abaixo com o comportamento especificado nas questões:

```python
# Modelos N-grama
construir_vocabulario(corpus) -> set
construir_modelo_bigrama(corpus) -> dict
calcular_probabilidade_bigrama(modelo, frase) -> float
gerar_texto_bigrama(modelo, palavra_inicial, n_palavras) -> str

# Sistema de Busca FAQ
criar_vetorizador_tfidf(corpus) -> TfidfVectorizer
vetorizar_texto(texto, vetorizador) -> sparse matrix
calcular_similaridade(vetor1, vetor2) -> float
buscar_faq(pergunta, perguntas_faq, respostas_faq, vetorizador, threshold) -> tuple
```

Notas:
- A implementação é avaliada pelo **comportamento observável**: saídas, tipos e chaves retornadas.
- Este notebook impõe técnicas específicas por design, pois o resultado será consumido por etapas posteriores.

## Setup

Esta seção prepara o ambiente e carrega dependências e dados do problema.

Requisitos:
- Execute as células desta seção **sem alterações**, exceto quando o notebook indicar explicitamente o contrário.
- Se houver erro de importação/instalação, corrija o ambiente (não altere a assinatura das funções avaliadas em nenhuma hipótese).

In [1]:
# === SETUP ===
!pip install nltk scikit-learn pandas --quiet

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

import random
import math
from collections import Counter, defaultdict
from nltk.tokenize import word_tokenize
from nltk import bigrams, trigrams
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

SEED_AVALIACAO = 42
np.random.seed(SEED_AVALIACAO)
random.seed(SEED_AVALIACAO)

# Indices especificos de validacao - SERAO USADOS NOS TESTES
FAQ_VALIDACAO_Q8_EXATA_IDX = 5
FAQ_VALIDACAO_Q8_SIMILAR_IDX = 12
INDICES_VALIDACAO_Q9 = [2, 7, 15, 23, 31]

print("==> Setup completo!")

==> Setup completo!


In [2]:
# === CARREGAR DATASET DE FAQ ===
# Não modifique esta célula

faq_data = {
    'categoria': ['rastreamento', 'rastreamento', 'rastreamento', 'rastreamento', 'rastreamento',
                  'devolucao', 'devolucao', 'devolucao', 'devolucao', 'devolucao',
                  'pagamento', 'pagamento', 'pagamento', 'pagamento', 'pagamento',
                  'entrega', 'entrega', 'entrega', 'entrega', 'entrega',
                  'produto', 'produto', 'produto', 'produto', 'produto',
                  'conta', 'conta', 'conta', 'contato', 'contato'],
    'pergunta': [
        'Como rastrear meu pedido?',
        'Onde está minha encomenda?',
        'Meu pedido está atrasado o que fazer?',
        'Qual o prazo de entrega?',
        'O pedido saiu para entrega mas não chegou?',
        'Como faço para devolver um produto?',
        'Qual o prazo para devolução?',
        'O produto veio com defeito como proceder?',
        'Como funciona o reembolso?',
        'Posso trocar por outro produto?',
        'Quais formas de pagamento aceitas?',
        'Posso parcelar minha compra?',
        'O boleto venceu o que faço?',
        'Como pagar com Pix?',
        'Meu pagamento não foi aprovado?',
        'Entregam no Brasil todo?',
        'Posso alterar o endereço de entrega?',
        'Vocês entregam em caixa postal?',
        'Preciso estar em casa para receber?',
        'Qual transportadora vocês usam?',
        'O produto está disponível?',
        'Qual a garantia do produto?',
        'O produto é original?',
        'As cores do site são fiéis ao produto?',
        'O produto não é o que esperava?',
        'Como criar uma conta?',
        'Esqueci minha senha?',
        'Como alterar meus dados?',
        'Qual o horário de atendimento?',
        'Como falar com um atendente?'
    ],
    'resposta': [
        'Você pode rastrear seu pedido acessando Meus Pedidos no site ou app. Também enviamos atualizações por email a cada mudança de status.',
        'Para verificar a localização da sua encomenda acesse Meus Pedidos e clique em Rastrear. O código de rastreamento também pode ser consultado no site dos Correios.',
        'Se seu pedido ultrapassou o prazo de entrega entre em contato conosco informando o número do pedido. Vamos verificar junto à transportadora e te dar um retorno em até 24h.',
        'O prazo de entrega varia conforme sua região e o tipo de frete escolhido. Frete normal de 5 a 15 dias úteis. Frete expresso de 1 a 5 dias úteis.',
        'Quando o status indica Saiu para entrega a encomenda deve chegar no mesmo dia ou no próximo dia útil. Se não receber entre em contato conosco.',
        'Você tem até 7 dias após o recebimento para solicitar a devolução. Acesse Meus Pedidos selecione o item e clique em Solicitar Devolução.',
        'O prazo para solicitar devolução é de 7 dias corridos após o recebimento do produto conforme o Código de Defesa do Consumidor.',
        'Para produtos com defeito você pode solicitar troca ou reembolso em até 30 dias. Acesse Meus Pedidos e selecione Produto com Defeito.',
        'Após recebermos e verificarmos o produto devolvido o reembolso é processado em até 10 dias úteis. O valor retorna para a mesma forma de pagamento.',
        'Sim você pode solicitar troca por outro produto do mesmo valor ou pagar a diferença se for mais caro.',
        'Aceitamos cartão de crédito Visa Mastercard Elo American Express, cartão de débito, boleto bancário e Pix.',
        'Sim compras acima de R$50 podem ser parceladas em até 12x no cartão de crédito. Parcelas mínimas de R$20.',
        'Boletos vencidos não podem ser pagos. Você pode gerar um novo boleto acessando Meus Pedidos.',
        'Ao finalizar a compra selecione Pix como forma de pagamento. Um QR Code será exibido e também um código para copiar e colar.',
        'Se o pagamento não foi aprovado verifique os dados do cartão, limite disponível e se não há bloqueio no banco.',
        'Sim entregamos para todo o Brasil. Algumas regiões remotas podem ter prazo de entrega estendido.',
        'O endereço pode ser alterado apenas se o pedido ainda não foi enviado. Acesse Meus Pedidos ou entre em contato conosco.',
        'Não realizamos entregas em caixas postais. É necessário informar um endereço residencial ou comercial.',
        'Recomendamos que tenha alguém para receber. Caso não tenha a transportadora fará novas tentativas.',
        'Trabalhamos com Correios e transportadoras parceiras como Jadlog e Total Express dependendo da sua região.',
        'A disponibilidade é mostrada na página do produto. Se aparecer Avise-me quando chegar significa que está esgotado.',
        'Todos os produtos têm garantia legal de 90 dias. Alguns produtos possuem garantia estendida do fabricante.',
        'Sim todos os nossos produtos são originais com nota fiscal. Trabalhamos apenas com fornecedores autorizados.',
        'Fazemos o possível para que as fotos representem fielmente os produtos porém pode haver pequenas variações.',
        'Se o produto não atendeu suas expectativas você pode devolvê-lo em até 7 dias desde que esteja lacrado.',
        'Clique em Criar Conta no topo do site e preencha seus dados. Você também pode criar durante o checkout.',
        'Clique em Esqueci minha senha na página de login. Enviaremos um link para redefinição no seu email cadastrado.',
        'Acesse Minha Conta e clique em Dados Pessoais para atualizar nome, email, telefone ou endereço.',
        'Nosso atendimento funciona de segunda a sexta das 8h às 20h e sábados das 8h às 14h.',
        'Você pode falar conosco pelo chat no site, telefone 0800-XXX-XXXX ou email atendimento@loja.com.br.'
    ]
}

df_faq = pd.DataFrame(faq_data)
perguntas_faq = df_faq['pergunta'].tolist()
respostas_faq = df_faq['resposta'].tolist()

print(f"==> FAQ carregado: {len(df_faq)} perguntas/respostas")
print(f"\nCategorias: {df_faq['categoria'].unique().tolist()}")

==> FAQ carregado: 30 perguntas/respostas

Categorias: ['rastreamento', 'devolucao', 'pagamento', 'entrega', 'produto', 'conta', 'contato']


---

## Parte 1: Modelos de Linguagem N-grama (40 pontos)

Você irá construir um modelo de linguagem por **bigramas** a partir de um pequeno corpus textual.
O objetivo não é "alta qualidade" de geração, e sim implementar corretamente:

- extração de vocabulário
- contagens de transição (bigrama)
- cálculo de probabilidade de uma frase
- geração estocástica baseada em contagens

Restrições gerais:
- O tokenizador e o pré-processamento devem ser consistentes entre as funções quando aplicável (minúsculas, filtragem de tokens não alfabéticos).
- Preserve assinaturas e valores default.

In [3]:
# Corpus para treinar o modelo de linguagem
corpus_reviews = [
    "o produto é muito bom recomendo a todos",
    "o produto chegou rápido e bem embalado",
    "gostei muito do produto excelente qualidade",
    "o produto é excelente superou minhas expectativas",
    "produto de ótima qualidade entrega rápida",
    "o produto é bom mas a entrega demorou",
    "muito satisfeito com o produto recomendo",
    "produto bom pelo preço que paguei",
    "a qualidade do produto é muito boa",
    "produto chegou antes do prazo muito bom",
    "excelente produto entrega muito rápida",
    "o produto é bom custo benefício excelente",
    "gostei do produto vou comprar novamente",
    "produto de qualidade preço justo",
    "muito bom o produto funcionou perfeitamente",
    "o produto atendeu minhas expectativas",
    "ótimo produto recomendo a compra",
    "produto excelente chegou muito rápido",
    "a entrega foi rápida produto muito bom",
    "produto de boa qualidade preço acessível"
]
print(f"Corpus: {len(corpus_reviews)} textos")

Corpus: 20 textos


### Questão 1: Construir Vocabulário (10 pontos)

### Objetivo
Construir um vocabulário (conjunto de termos) a partir de um corpus de textos, incluindo tokens especiais exigidos pela avaliação.

### Entradas esperadas
- `corpus: list[str]` contendo documentos de texto.
- O corpus já vem em português e deve ser tratado como texto livre.

### Processamento obrigatório
Implemente `construir_vocabulario(corpus)` de forma que:

1. Inicialize um `set` de vocabulário contendo **obrigatoriamente** os tokens especiais:
   - `<START>`
   - `<END>`
2. Para cada documento do corpus:
   - converta para minúsculas;
   - tokenize;
   - mantenha apenas tokens alfabéticos.
3. Adicione todos os tokens válidos ao conjunto.
4. Retorne o conjunto final.

### Artefatos que devem existir ao final
Ao final desta questão, deve existir:

- Função `construir_vocabulario(corpus: list) -> set`
- Uma variável `vocabulario` (ou equivalente no teste local) contendo o retorno da função quando chamada com o corpus fornecido, com:
  - tipo `set`
  - presença de `'produto'` (dado o corpus de treino)
  - presença de `<START>` e `<END>`.

### Restrições técnicas
- Não retorne lista nem dict: deve ser `set`.
- Não remova os tokens especiais.
- Não altere o nome da função nem a ordem dos parâmetros.

### Observações de continuidade
Este vocabulário é um artefato de suporte para validação do pipeline de N-gramas e para consistência com etapas posteriores.

In [6]:
def construir_vocabulario(corpus: list) -> set:
    """
    Constrói vocabulário a partir de um corpus.

    Args:
        corpus: Lista de textos (strings)

    Returns:
        Conjunto com todas as palavras únicas + <START> e <END>
    """
    # === SEU CÓDIGO AQUI ===
    vocab = {"<START>", "<END>"}
    for doc in corpus:
        tokens = word_tokenize(doc.lower())
        tokens = [t for t in tokens if t.isalpha()]
        vocab.update(tokens)
    return vocab



In [7]:
# Teste
vocabulario = construir_vocabulario(corpus_reviews)
print(f"Tamanho: {len(vocabulario)}")
assert '<START>' in vocabulario and '<END>' in vocabulario
print("OK")

Tamanho: 50
OK


### Questão 2: Construir Modelo Bigrama (10 pontos)

### Objetivo
Construir um modelo de bigramas baseado em contagens de transição (palavra anterior -> próxima palavra), que será usado tanto para probabilidade quanto para geração.

### Entradas esperadas
- `corpus: list[str]` contendo documentos de treino.
- O pré-processamento deve ser consistente com a Questão 1 (minúsculas e filtragem alfabética).

### Processamento obrigatório
Implemente `construir_modelo_bigrama(corpus)` de forma que:

1. Para cada documento:
   - tokenize em minúsculas;
   - filtre tokens não alfabéticos;
   - adicione marcadores de início/fim **nesta função** como:
     - `<s>` no início
     - `</s>` no fim
2. Gere bigramas sobre a sequência tokenizada.
3. Retorne um `dict` normal (não `defaultdict`) no formato:
   - `dict[str, dict[str, int]]`.

### Artefatos que devem existir ao final
Ao final desta questão, deve existir:

- Função `construir_modelo_bigrama(corpus: list) -> dict`
- A estrutura retornada deve conter a chave `<s>` como token inicial.


### Restrições técnicas
- O retorno deve ser serializável como `dict` (não deixe `defaultdict` no retorno).
- Não mude os marcadores `<s>` e `</s>`; eles serão usados implicitamente nas próximas questões.
- Não normalize para probabilidades aqui; apenas contagens.

### Observações de continuidade
Este modelo é consumido diretamente pelas Questões 3 e 4. Se a estrutura ou os marcadores forem alterados, as próximas funções deixam de ser compatíveis.

In [8]:
def construir_modelo_bigrama(corpus: list) -> dict:
    """
    Constrói modelo de linguagem bigrama.

    Args:
        corpus: Lista de textos

    Returns:
        Dict {palavra_anterior: {palavra: probabilidade}}
    """
    # === SEU CÓDIGO AQUI ===
    modelo = defaultdict(Counter)

    for doc in corpus:
        tokens = word_tokenize(doc.lower())
        tokens = [t for t in tokens if t.isalpha()]
        tokens = ["<s>"] + tokens + ["</s>"]

        for w1, w2 in bigrams(tokens):
            modelo[w1][w2] += 1

    return {k: dict(v) for k, v in modelo.items()}

In [9]:
# Teste
modelo_bigrama = construir_modelo_bigrama(corpus_reviews)
print(f"Q2: Modelo com {len(modelo_bigrama)} palavras iniciais")
print(f"    Continuações de 'produto': {modelo_bigrama.get('produto', {})}")
assert '<s>' in modelo_bigrama
print("✓ OK")

Q2: Modelo com 49 palavras iniciais
    Continuações de 'produto': {'é': 5, 'chegou': 2, 'excelente': 2, 'de': 3, 'recomendo': 2, 'bom': 1, 'entrega': 1, 'vou': 1, 'funcionou': 1, 'atendeu': 1, 'muito': 1}
✓ OK


### Questão 3: Calcular Log-Probabilidade de Frase (10 pontos)

### Objetivo
Calcular a log-probabilidade de uma frase segundo o modelo de bigramas por MLE, retornando `-inf` quando a frase contiver transições não vistas.

### Entradas esperadas
- `modelo: dict[str, dict[str, int]]` produzido por `construir_modelo_bigrama`.
- `frase: str` contendo uma frase de entrada.

### Processamento obrigatório
Implemente `calcular_probabilidade_bigrama(modelo, frase)` de forma que:

1. Tokenize `frase` em minúsculas e filtre tokens não alfabéticos.
2. Adicione marcadores:
   - `<s>` no início
   - `</s>` no fim
3. Calcule a log-probabilidade MLE da frase como a soma dos logs das probabilidades condicionais dos seus bigramas. Caso algum bigrama da frase não exista no modelo, a função deve retornar `float("-inf")`.
4. Retorne a log-probabilidade final (`float`).

### Artefatos que devem existir ao final
Ao final desta questão, deve existir:

- Função `calcular_probabilidade_bigrama(modelo: dict, frase: str) -> float`
- A função deve retornar um float representando log-probabilidade, tipicamente em `(-inf, 0.0]`.
- Para frases contendo tokens/transições fora do modelo, o retorno deve ser exatamente `float("-inf")`.

### Restrições técnicas
- Retorne log-probabilidade nesta entrega; o valor esperado é a soma de logs das probabilidades condicionais.
- Não aplique suavização (Laplace/Kneser-Ney etc.). O estimador aqui é MLE puro com fallback para `float("-inf")` em transições não vistas.
- Preserve assinaturas e nomes.

### Observações de continuidade
Este cálculo é usado para comparar frases plausíveis vs improváveis no corpus. A regra de retorno `-inf` para transições não vistas é usada como contrato nas validações.


In [10]:
def calcular_probabilidade_bigrama(modelo: dict, frase: str) -> float:
    """
    Calcula log-probabilidade de uma frase.

    Args:
        modelo: Modelo bigrama
        frase: Texto a avaliar

    Returns:
        Log-probabilidade (float negativo)
    """
    # === SEU CÓDIGO AQUI ===
    # Use 1e-10 para bigramas não vistos
    tokens = word_tokenize(frase.lower())
    tokens = [t for t in tokens if t.isalpha()]
    tokens = ["<s>"] + tokens + ["</s>"]

    log_prob = 0.0

    for w1, w2 in bigrams(tokens):
        if w1 not in modelo or w2 not in modelo[w1]:
            return float("-inf")

        total = sum(modelo[w1].values())
        prob = modelo[w1][w2] / total
        log_prob += math.log(prob)

    return log_prob

In [12]:
# Teste
modelo_bi = modelo_bigrama
lp1 = calcular_probabilidade_bigrama(modelo_bi, "produto muito bom")
lp2 = calcular_probabilidade_bigrama(modelo_bi, "xyz abc def")

print(f"logP('produto muito bom') = {lp1:.6f}")
print(f"logP('xyz abc def') = {lp2}")

assert lp1 > lp2  # maior logP = frase mais provável
print("✓ OK!")

logP('produto muito bom') = -6.263398
logP('xyz abc def') = -inf
✓ OK!


### Questão 4: Geração de Texto (10 pontos)

Implemente `gerar_texto_bigrama` usando amostragem ponderada.

### Objetivo
Gerar uma sequência de palavras amostrando a próxima palavra proporcionalmente às contagens do modelo (amostragem ponderada).

### Entradas esperadas
- `modelo: dict[str, dict[str, int]]` produzido por `construir_modelo_bigrama`.
- `palavra_inicial: str | None`:
  - se `None`, a geração deve iniciar em `<s>`;
  - caso contrário, a palavra inicial deve ser usada (em minúsculas).
- `n_palavras: int` número máximo de passos de geração (limite superior).

### Processamento obrigatório
Implemente `gerar_texto_bigrama(modelo, palavra_inicial, n_palavras)` de forma que:

1. Defina `palavra_atual`:
   - `<s>` se `palavra_inicial is None`
   - `palavra_inicial.lower()` caso contrário
2. Inicialize a lista de saída:
   - vazia se `palavra_atual == '<s>'`
   - contendo `palavra_atual` caso contrário
3. Repita até `n_palavras` iterações:
   - se `palavra_atual` não existir em `modelo`, encerre;
   - obtenha continuadores e contagens;
   - amostre `proxima` proporcionalmente às contagens (amostragem ponderada);
   - se `proxima == '</s>'`, encerre;
   - senão, adicione `proxima` e atualize `palavra_atual`.
4. Retorne a frase gerada como string, tokens separados por espaço.

### Artefatos que devem existir ao final
Ao final desta questão, deve existir:

- Função `gerar_texto_bigrama(modelo: dict, palavra_inicial: str, n_palavras: int) -> str`
- O retorno deve ser `str`.
- A geração deve parar ao encontrar `</s>` ou quando não houver transição possível.

### Restrições técnicas
- Use amostragem ponderada (não escolha sempre o máximo).
- Não inclua `<s>` e `</s>` no texto final.
- Preserve assinatura e comportamento de `palavra_inicial is None`.

### Observações de continuidade
Esta função é um artefato de demonstração e valida consistência do modelo bigrama. A mesma disciplina de tokens é pré-requisito para reuso no Projeto Final.

In [13]:
def gerar_texto_bigrama(modelo: dict, palavra_inicial: str = '<START>', n_palavras: int = 10) -> str:
    """
    Gera texto usando modelo bigrama.

    Args:
        modelo: Modelo bigrama
        palavra_inicial: Palavra para começar
        n_palavras: Máximo de palavras

    Returns:
        Texto gerado
    """
    # === SEU CÓDIGO AQUI ===
    # Use random.choices(lista, weights=probs)
    if palavra_inicial is None:
        palavra_atual = "<s>"
        resultado = []
    else:
        palavra_atual = palavra_inicial.lower()
        resultado = [palavra_atual]

    for _ in range(n_palavras):
        if palavra_atual not in modelo:
            break

        proximas = list(modelo[palavra_atual].keys())
        pesos = list(modelo[palavra_atual].values())

        proxima = random.choices(proximas, weights=pesos, k=1)[0]

        if proxima == "</s>":
            break

        resultado.append(proxima)
        palavra_atual = proxima

    return " ".join(resultado)

In [14]:
# Teste
random.seed(SEED_AVALIACAO)
texto1 = gerar_texto_bigrama(modelo_bi, 'produto', 8)
texto2 = gerar_texto_bigrama(modelo_bi, None, 8)
print(f"Gerado de 'produto': {texto1}")
print(f"Gerado de <s>: {texto2}")
assert isinstance(texto1, str) and len(texto1) > 0
print("✓ OK!")

Gerado de 'produto': produto recomendo a entrega rápida produto recomendo
Gerado de <s>: o produto é muito do produto é bom
✓ OK!


---

## Parte 2: Sistema de Busca FAQ com TF-IDF (60 pontos)

Nesta parte você implementará um mecanismo de recuperação de respostas de FAQ usando:

- Vetorização **TF-IDF** (com parâmetros fixos)
- **Similaridade cosseno**
- Seleção do melhor candidato com limiar (threshold)

Restrições gerais:
- O vetorizador deve ser treinado a partir das perguntas da base.
- A função `buscar_faq` é considerada **crítica**: ela será consumida diretamente no Projeto Final.



### Questão 5: Criar Vetorizador TF-IDF (10 pontos)

### Objetivo
Criar um `TfidfVectorizer` com hiperparâmetros impostos e treiná-lo sobre o corpus de perguntas.

### Entradas esperadas
- `corpus: list[str]` contendo textos (perguntas da FAQ).

### Processamento obrigatório
Implemente `criar_vetorizador_tfidf(corpus)` de forma que:

1. Instancie `TfidfVectorizer` com os seguintes parâmetros:
   - `ngram_range=(1, 2)`
   - `max_features=500`
   - `min_df=1`
   - `max_df=0.95`
   - `lowercase=True`
   - `strip_accents='unicode'`
2. Faça `fit(corpus)`.
3. Retorne o objeto vetorizador treinado.

### Artefatos que devem existir ao final
Ao final desta questão, deve existir:

- Função `criar_vetorizador_tfidf(corpus: list) -> TfidfVectorizer`
- O retorno deve possuir o atributo `vocabulary_` após o `fit`.
- `vetorizador.get_feature_names_out()` deve funcionar.

### Restrições técnicas
- Não altere os hiperparâmetros. Eles fazem parte do contrato do pipeline cumulativo.
- Não use outros vetorizadores (CountVectorizer etc.).

### Observações de continuidade
O mesmo vetorizador (e seu vocabulário) será usado para vetorizar perguntas do usuário e a base de FAQ, garantindo compatibilidade na comparação.

In [15]:
def criar_vetorizador_tfidf(corpus: list) -> TfidfVectorizer:
    """
    Cria e treina vetorizador TF-IDF.
    Use ngram_range=(1,2) e min_df=1.

    Args:
        corpus: Lista de textos

    Returns:
        TfidfVectorizer treinado
    """
    # === SEU CÓDIGO AQUI ===
    vet = TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=500,
        min_df=1,
        max_df=0.95,
        lowercase=True,
        strip_accents='unicode'
    )
    vet.fit(corpus)
    return vet

In [16]:
# Teste
vetorizador = criar_vetorizador_tfidf(perguntas_faq)
print(f"Vocabulário TF-IDF com {len(vetorizador.vocabulary_)} termos")
assert isinstance(vetorizador, TfidfVectorizer)
print("✓ OK!")

Vocabulário TF-IDF com 188 termos
✓ OK!


### Questão 6: Vetorizar Texto (10 pontos)

### Objetivo
Transformar um texto arbitrário em um vetor TF-IDF compatível com o vocabulário do vetorizador treinado.

### Entradas esperadas
- `texto: str` com a pergunta do usuário ou uma pergunta da base.
- `vetorizador: TfidfVectorizer` já treinado (Questão 5).

### Processamento obrigatório
Implemente `vetorizar_texto(texto, vetorizador)` de forma que:

1. Use `vetorizador.transform([texto])` (lista com um único elemento).
2. Retorne o vetor esparso resultante.

Observação: o retorno esperado é uma matriz esparsa com shape `(1, |V|)`.

### Artefatos que devem existir ao final
Ao final desta questão, deve existir:

- Função `vetorizar_texto(texto: str, vetorizador: TfidfVectorizer)`
- Para qualquer string, o retorno deve ter:
  - `shape[0] == 1`

### Restrições técnicas
- Não use `fit_transform` aqui (isso mudaria o modelo).
- Não retorne vetor denso (evite `.toarray()` no retorno).
- Preserve assinatura e nomes.

### Observações de continuidade
Esta função padroniza a forma de entrada do pipeline: toda pergunta do usuário passa por ela antes de similaridade e busca.


In [17]:
def vetorizar_texto(texto: str, vetorizador: TfidfVectorizer):
    """
    Transforma texto em vetor TF-IDF.

    Args:
        texto: String a vetorizar
        vetorizador: TfidfVectorizer treinado

    Returns:
        Matriz esparsa (1, n_features)
    """
    # === SEU CÓDIGO AQUI ===
    return vetorizador.transform([texto])

In [18]:
# Teste
vetor = vetorizar_texto("como rastrear pedido", vetorizador)
print(f"Shape: {vetor.shape}, elementos não-zero: {vetor.nnz}")
assert vetor.shape[0] == 1
print("✓ OK")

Shape: (1, 188), elementos não-zero: 4
✓ OK


### Questão 7: Calcular Similaridade (10 pontos)

### Objetivo
Calcular a similaridade cosseno entre dois vetores TF-IDF, retornando um escalar `float` em `[0, 1]`.

### Entradas esperadas
- `vetor1`: matriz esparsa 1×|V| (por exemplo, saída de `vetorizar_texto`).
- `vetor2`: matriz esparsa 1×|V| (mesma dimensionalidade).

### Processamento obrigatório
Implemente `calcular_similaridade(vetor1, vetor2)` usando a similaridade cosseno.

### Artefatos que devem existir ao final
Ao final desta questão, deve existir:

- Função `calcular_similaridade(vetor1, vetor2) -> float`
- O retorno deve respeitar:
  - `0.0 <= sim <= 1.0` em casos usuais com TF-IDF.

### Restrições técnicas
- Garanta que o retorno seja `float`, não array NumPy.

### Observações de continuidade
Este cálculo é o núcleo da recuperação por similaridade e será usado direta ou indiretamente na busca final.

In [19]:
def calcular_similaridade(vetor1, vetor2) -> float:
    """
    Calcula similaridade cosseno entre dois vetores.

    Args:
        vetor1, vetor2: Matrizes esparsas

    Returns:
        Similaridade (float entre 0 e 1)
    """
    # === SEU CÓDIGO AQUI ===
    sim = cosine_similarity(vetor1, vetor2)[0, 0]
    return float(sim)

In [20]:
# Teste
v1 = vetorizar_texto("como rastrear pedido", vetorizador)
v2 = vetorizar_texto("rastrear meu pedido", vetorizador)
v3 = vetorizar_texto("formas de pagamento", vetorizador)
sim12 = calcular_similaridade(v1, v2)
sim13 = calcular_similaridade(v1, v3)
print(f"Sim(rastrear, rastrear) = {sim12:.4f}")
print(f"Sim(rastrear, pagamento) = {sim13:.4f}")
assert sim12 > sim13
print("✓ OK!")

Sim(rastrear, rastrear) = 0.4714
Sim(rastrear, pagamento) = 0.0000
✓ OK!


### Questão 8: Sistema de Busca FAQ (15 pontos)


### Objetivo
Implementar a função principal de recuperação: dado um texto de pergunta, retornar a melhor resposta da base, condicionada a um limiar mínimo de similaridade.

### Entradas esperadas
- `pergunta: str` (entrada do usuário).
- `perguntas_faq: list[str]` (base de perguntas).
- `respostas_faq: list[str]` (base de respostas), alinhada por índice com `perguntas_faq`.
- `vetorizador: TfidfVectorizer` treinado sobre perguntas da FAQ.
- `threshold: float` (default `0.3`), limiar mínimo de aceitação.

### Processamento obrigatório
Implemente `buscar_faq(...)` de forma que retorne a resposta da FAQ mais similar à pergunta, juntamente com o valor da similaridade cosseno, ou `(None, 0.0)` caso nenhuma similaridade atinja o limiar (threshold) definido.

### Artefatos que devem existir ao final
Ao final desta questão, deve existir:

- Função `buscar_faq(pergunta, perguntas_faq, respostas_faq, vetorizador, threshold=0.3) -> tuple`
- O retorno deve ser **sempre** uma tupla de tamanho 2:
  - `resposta: str | None`
  - `score: float`
- `score` deve ser `0.0` quando não houver candidato aceito pelo limiar.

### Restrições técnicas
- Preserve o valor default de `threshold` em `0.3`.
- Não altere a regra de desempate: escolha o maior score.
- Não re-treine vetorizador dentro desta função.
- Garanta alinhamento por índice entre perguntas e respostas.

### Observações de continuidade
Esta função é consumida no Projeto Final. O contrato de retorno (tupla com `(resposta|None, score)`) deve ser preservado para integração com o chatbot.

In [21]:
def buscar_faq(pergunta: str,
               perguntas_faq: list,
               respostas_faq: list,
               vetorizador: TfidfVectorizer,
               threshold: float = 0.3) -> tuple:
    """
    Busca a resposta mais relevante no FAQ.

    Args:
        pergunta: Pergunta do usuário
        perguntas_faq: Lista de perguntas do FAQ
        respostas_faq: Lista de respostas
        vetorizador: TfidfVectorizer treinado
        threshold: Similaridade mínima

    Returns:
        (resposta, score) ou (None, 0.0)
    """
    # === SEU CÓDIGO AQUI ===
    vetor_pergunta = vetorizar_texto(pergunta, vetorizador)
    vetores_faq = vetorizador.transform(perguntas_faq)

    similaridades = cosine_similarity(vetor_pergunta, vetores_faq)[0]
    idx_max = int(np.argmax(similaridades))
    score_max = float(similaridades[idx_max])

    if score_max >= threshold:
        return respostas_faq[idx_max], score_max
    else:
        return None, 0.0

In [22]:
# Teste
respostas_faq = df_faq['resposta'].tolist()

testes = [
    "como rastrear minha compra",
    "quero devolver produto",
    "xyz abc 123"
]

for p in testes:
    resp, score = buscar_faq(p, perguntas_faq, respostas_faq, vetorizador, 0.2)
    print(f"\n'{p}'")
    print(f"  Score: {score:.4f}")
    print(f"  Resp: {resp[:60] if resp else 'Não encontrada'}...")

print("\n✓ OK")


'como rastrear minha compra'
  Score: 0.4726
  Resp: Sim compras acima de R$50 podem ser parceladas em até 12x no...

'quero devolver produto'
  Score: 0.3911
  Resp: Você tem até 7 dias após o recebimento para solicitar a devo...

'xyz abc 123'
  Score: 0.0000
  Resp: Não encontrada...

✓ OK


### Questão 9: Sistema FAQ Completo (15 pontos)

### Objetivo
Criar uma função de inicialização que consolida os artefatos do sistema de FAQ em uma estrutura única (dicionário) para reuso em etapas posteriores.

### Entradas esperadas
- `perguntas: list[str]` (perguntas da base).
- `respostas: list[str]` (respostas da base), alinhadas por índice.

### Processamento obrigatório
Implemente `criar_sistema_faq(perguntas, respostas)` de forma que:

1. Crie e treine o vetorizador chamando `criar_vetorizador_tfidf(perguntas)`.
2. Vetorize todas as perguntas.   
3. Retorne um `dict` com as seguintes chaves obrigatórias:
   - `'vetorizador'`: o objeto treinado
   - `'vetores_faq'`: matriz esparsa com todos os vetores da base
   - `'perguntas'`: a lista de perguntas original
   - `'respostas'`: a lista de respostas original

### Artefatos que devem existir ao final
Ao final desta questão, deve existir:

- Função `criar_sistema_faq(perguntas: list, respostas: list) -> dict`
- O dicionário retornado deve conter **todas** as chaves:
  - `['vetorizador', 'vetores_faq', 'perguntas', 'respostas']`
- `vetores_faq.shape[0]` deve ser igual a `len(perguntas)`.

### Restrições técnicas
- Não renomeie chaves.
- Não retorne classes customizadas; o contrato é `dict`.
- Não faça cópias desnecessárias de dados grandes (matriz esparsa deve permanecer esparsa).

### Observações de continuidade
Este dicionário é a forma padrão de inicialização do componente de FAQ no Projeto Final, evitando recomputação e simplificando integração.


In [23]:
def criar_sistema_faq(perguntas: list, respostas: list) -> dict:
    """
    Cria sistema de FAQ completo.

    Args:
        perguntas: Lista de perguntas
        respostas: Lista de respostas

    Returns:
        Dict com 'vetorizador', 'vetores_faq', 'perguntas', 'respostas'
    """
    # === SEU CÓDIGO AQUI ===
    vetorizador = criar_vetorizador_tfidf(perguntas)
    vetores_faq = vetorizador.transform(perguntas)

    sistema = {
        "vetorizador": vetorizador,
        "vetores_faq": vetores_faq,
        "perguntas": perguntas,
        "respostas": respostas
    }

    return sistema

In [24]:
# Teste
sistema_faq = criar_sistema_faq(perguntas_faq, respostas_faq)
print(f"Perguntas: {len(sistema_faq['perguntas'])}")
print(f"Vetores shape: {sistema_faq['vetores_faq'].shape}")
assert all(k in sistema_faq for k in ['vetorizador', 'vetores_faq', 'perguntas', 'respostas'])
print("✓ OK")

Perguntas: 30
Vetores shape: (30, 188)
✓ OK


---

## Demonstração: Chatbot FAQ

Esta seção valida a integração mínima do sistema implementado.

Requisitos:
- Execute a demonstração para verificar que:
  - o sistema retorna respostas coerentes para perguntas semelhantes às da base;
  - o limiar (`threshold`) controla corretamente quando retornar `None`.
- Se o comportamento observado for inconsistente, trate como defeito de contrato em uma das funções de Q5–Q9.

In [25]:
print("DEMONSTRAÇÃO DO CHATBOT FAQ")
print("="*60)

perguntas_demo = [
    "como posso rastrear meu pedido?",
    "quero devolver um produto com defeito",
    "vocês parcelam em quantas vezes?",
    "meu pedido atrasou muito",
    "quero falar com atendente"
]

for pergunta in perguntas_demo:
    resposta, score = buscar_faq(
        pergunta,
        sistema_faq['perguntas'],
        sistema_faq['respostas'],
        sistema_faq['vetorizador'],
        0.2
    )

    print(f"\n👤 {pergunta}")
    print(f"   [Score: {score:.2f}]")
    if resposta:
        print(f"🤖 {resposta[:100]}...")
    else:
        print("🤖 Não encontrei resposta.")

DEMONSTRAÇÃO DO CHATBOT FAQ

👤 como posso rastrear meu pedido?
   [Score: 0.84]
🤖 Você pode rastrear seu pedido acessando Meus Pedidos no site ou app. Também enviamos atualizações po...

👤 quero devolver um produto com defeito
   [Score: 0.53]
🤖 Você tem até 7 dias após o recebimento para solicitar a devolução. Acesse Meus Pedidos selecione o i...

👤 vocês parcelam em quantas vezes?
   [Score: 0.44]
🤖 Não realizamos entregas em caixas postais. É necessário informar um endereço residencial ou comercia...

👤 meu pedido atrasou muito
   [Score: 0.62]
🤖 Você pode rastrear seu pedido acessando Meus Pedidos no site ou app. Também enviamos atualizações po...

👤 quero falar com atendente
   [Score: 0.68]
🤖 Você pode falar conosco pelo chat no site, telefone 0800-XXX-XXXX ou email atendimento@loja.com.br....


---

## Resumo - Funções para o Projeto Final

As seguintes funções e artefatos serão consumidos diretamente na próxima etapa:

| Função / Artefato | Responsabilidade |
|---|---|
| `criar_vetorizador_tfidf()` | Definir e treinar o modelo de representação |
| `vetorizar_texto()` | Normalizar a entrada do usuário para o espaço vetorial |
| `calcular_similaridade()` | Medir proximidade no espaço TF-IDF |
| `buscar_faq()` | Selecionar a melhor resposta com limiar |
| `criar_sistema_faq()` | Inicializar e empacotar artefatos do sistema |

Requisito de estabilidade:
- Não altere assinaturas, chaves retornadas ou tipos: isso é parte do contrato de integração.

---

## Checklist de Entrega

## Checklist de submissão

Antes de submeter, valide objetivamente:

- [ ] Todas as funções estão implementadas e executam sem exceções.
- [ ] O notebook executa do início ao fim, em ordem, sem intervenção manual.
- [ ] As funções críticas (Q5–Q9) preservam:
  - assinatura
  - tipos de retorno
  - chaves obrigatórias no dicionário do sistema
  - valores default (especialmente `threshold=0.3`)
- [ ] A demonstração final executa e retorna saídas compatíveis com a especificação.

**Envie via Moodle até a data limite.**